# Forex_DNN Framework: Hybrid Rule-Based + Machine Learning Workbench

This notebook demonstrates the end-to-end integration of the newly implemented **Feature Registry**, **Feature Pipeline**, **Dataset Builder**, **Data Cleaner**, and the **ML Baseline Models** (Market State Classifier and Level Break Probability Model).

## Notebook Structure
1. **Setup & Initialization**: Loading the Feature Registry and verifying metadata.
2. **Data Generation**: Creating synthetic EURUSD M5 historical data containing indicators and market structures.
3. **Dataset Building**: Constructing Dataset A (Market State) and Dataset B (Level Break) from historical structures.
4. **Data Cleaning & Validation**: Standardizing inputs, dropping duplicates, handling missing values, and generating a dataset quality report.
5. **Baseline Model Training**: Training both LightGBM classifiers, logging performance metrics, and extracting feature importances.
6. **Backtesting / Inference**: Demonstrating prediction-only ML model execution integrated into `MMStrategy`.

## 1. Setup & Initialization

In [6]:
import os
import sys

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
from ML.feature_registry import FeatureRegistry
from ML.feature_pipeline import FeaturePipeline
from ML.dataset_builder import DatasetBuilder
from ML.data_cleaner import DataCleaner

# Initialize centralized Feature Registry
registry = FeatureRegistry(load_defaults=True)
print(f"Feature Registry loaded. Total registered features: {len(registry.list_all())}")
print(f"Enabled features: {len(registry.list_enabled())}")
print(f"Registry Version Hash: {registry.compute_hash()[:12]}")

# Auto-generate registry documentation markdown table
registry.export_markdown("../docs/feature_registry.md")
registry.generate_visualization("../reports/feature_map.html")
print("Documentation & Interactive Map generated successfully.")

Feature Registry loaded. Total registered features: 49
Enabled features: 49
Registry Version Hash: 26e43c8b6b23
Documentation & Interactive Map generated successfully.


## 2. Mocking EURUSD M5 Data & Structures

In [8]:
from Market_Data_Pipeline.structure_graph import MarketStructureGraph, StructureLevel, Zone, BOS, CHOCH
from datetime import datetime, timedelta

# Create synthetic DataFrame containing required indicators
n_samples = 1200
np.random.seed(42)

datetimes = [datetime(2024, 1, 1) + timedelta(minutes=5 * i) for i in range(n_samples)]
prices = 1.1000 + np.cumsum(np.random.randn(n_samples) * 0.0001)

df = pd.DataFrame({
    "Datetime": datetimes,
    "Open": prices + np.random.randn(n_samples) * 0.0001,
    "High": prices + np.abs(np.random.randn(n_samples)) * 0.0002,
    "Low": prices - np.abs(np.random.randn(n_samples)) * 0.0002,
    "Close": prices,
    "TickVolume": np.random.randint(100, 1000, n_samples),
    "Spread": np.ones(n_samples) * 1.5,
    "ema_50": prices + np.random.randn(n_samples) * 0.0005,
    "ema_600": prices + np.random.randn(n_samples) * 0.0010,
    "atr_14": np.random.uniform(0.0001, 0.0003, n_samples),
    "body_pct": np.random.uniform(0.1, 0.9, n_samples),
    "candle_size": np.random.uniform(0.0002, 0.0008, n_samples),
    "upper_shadow": np.random.uniform(0.0, 0.0002, n_samples),
    "lower_shadow": np.random.uniform(0.0, 0.0002, n_samples),
    "ema_slope_50": np.random.uniform(0.0, 1.5, n_samples),
    "ema_slope_600": np.random.uniform(0.0, 1.5, n_samples),
    "dist_ema_50": np.random.randn(n_samples)
})

# Build corresponding MarketStructureGraph with synthetic zones and breaks
msg = MarketStructureGraph(
    symbol="EURUSD_o",
    timeframe="M5",
    timestamp=df["Datetime"].iloc[-1]
)

# Inject standard swings, BOS and CHOCH
msg.swing_highs = [StructureLevel(price=df.iloc[idx]["High"], index=idx, level_type="SwingHigh") for idx in range(50, n_samples, 80)]
msg.swing_lows = [StructureLevel(price=df.iloc[idx]["Low"], index=idx, level_type="SwingLow") for idx in range(10, n_samples, 80)]
msg.bos = [BOS(index=idx, direction=1 if idx % 200 == 0 else -1, broken_level=df.iloc[idx]["Close"]) for idx in range(100, n_samples, 150)]
msg.choch = [CHOCH(index=idx, previous_trend=-1, new_trend=1, price=df.iloc[idx]["Close"]) for idx in range(120, n_samples, 250)]

# Inject Supply/Demand Zones
msg.supply_zones = [
    Zone(upper=prices[200] + 0.0010, lower=prices[200] + 0.0005, type="Supply", created_idx=200, freshness=True, strength_score=2.0),
    Zone(upper=prices[600] + 0.0010, lower=prices[600] + 0.0005, type="Supply", created_idx=600, freshness=True, strength_score=1.5)
]
msg.demand_zones = [
    Zone(upper=prices[100] - 0.0005, lower=prices[100] - 0.0010, type="Demand", created_idx=100, freshness=True, strength_score=2.5),
    Zone(upper=prices[500] - 0.0005, lower=prices[500] - 0.0010, type="Demand", created_idx=500, freshness=True, strength_score=1.8)
]

print(f"MarketStructureGraph initialized with {len(msg.swing_highs)} Swing Highs, {len(msg.bos)} BOS, and {len(msg.supply_zones)+len(msg.demand_zones)} S/D Zones.")

MarketStructureGraph initialized with 15 Swing Highs, 8 BOS, and 4 S/D Zones.


## 3. Constructing Datasets

In [9]:
builder = DatasetBuilder(registry)

# Build Dataset A: Market State Dataset
ds_state = builder.build_market_state_dataset(df, msg)
print(f"Dataset A (Market State) constructed. Rows: {len(ds_state)}")
print("Sample State Labels count:")
print(ds_state["label"].value_counts())

# Build Dataset B: Level Break Dataset
ds_break = builder.build_level_break_dataset(df, msg, lookahead_bars=25)
print(f"\nDataset B (Level Break) constructed. Rows: {len(ds_break)}")
if not ds_break.empty:
    print("Sample Target Class count:")
    print(ds_break["target"].value_counts())

Dataset A (Market State) constructed. Rows: 1100
Sample State Labels count:
label
TREND         590
TRANSITION    260
RANGE         250
Name: count, dtype: int64

Dataset B (Level Break) constructed. Rows: 60
Sample Target Class count:
target
0    49
1    11
Name: count, dtype: int64


## 4. Cleaning & Validation Reports

In [10]:
cleaner = DataCleaner()

# Clean Market State Dataset
cleaned_state = cleaner.clean(ds_state, label_col="label")

# Generate split & report
split_idx = int(len(cleaned_state) * 0.8)
df_train = cleaned_state.iloc[:split_idx]
df_test = cleaned_state.iloc[split_idx:]

cleaner.generate_report(df_train, df_test, label_col="label", filepath="../reports/dataset_report.html")
print("Data cleaning and dataset report generation completed successfully!")

Data cleaning and dataset report generation completed successfully!


## 5. Model Training

In [11]:
from sklearn.metrics import classification_report, confusion_matrix
from ML.models.market_state_classifier import MarketStateClassifier

feature_cols = [c for c in cleaned_state.columns if c not in ["label", "confidence", "timestamp"]]
X_train, X_test = df_train[feature_cols], df_test[feature_cols]

# Encode labels to integer indices (TREND=0, RANGE=1, TRANSITION=2)
class_map = {"TREND": 0, "RANGE": 1, "TRANSITION": 2}
y_train = df_train["label"].map(class_map).to_numpy()
y_test = df_test["label"].map(class_map).to_numpy()

# Train baseline LightGBM / RandomForest wrapper classifier
msc = MarketStateClassifier(model_type="lightgbm")
msc.fit(X_train, y_train)

# Evaluate performance
y_pred = msc.model.predict(X_test)
print("Market State Classifier Performance Results:")
print(classification_report(y_test, y_pred, target_names=["TREND", "RANGE", "TRANSITION"]))

# Feature importance
importances = msc.get_feature_importance()
sorted_imp = sorted(importances.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 Feature Importances:")
for name, score in sorted_imp[:10]:
    print(f"  {name:<30}: {score:.4f}")

# Save the trained model artifact
msc.save("../output/market_state_classifier.joblib")

ModuleNotFoundError: No module named 'ML.models'

## 6. Backtesting & Prediction-Only Simulation

In [ ]:
from Strategies.mm_strategy import MMStrategy
from Visualization.chart_annotator import ChartAnnotationEngine
from unittest.mock import MagicMock

# Instantiate the Strategy with our ML models loaded
data_feed = MagicMock()
send_order = MagicMock()
journal = MagicMock()
drawdown = MagicMock()
annotator = ChartAnnotationEngine()

strategy = MMStrategy(
    data_feed=data_feed,
    send_order=send_order,
    trading_journal=journal,
    drawdown_manager=drawdown,
    symbols=["EURUSD_o"],
    annotator=annotator,
    market_state_model=msc
)

print("MMStrategy initialized successfully in Hybrid Rule-Based + Machine Learning Mode!")
print(f"Attached Market State Classifier model: {strategy.market_state_model is not None}")